# Assignment 4: Reinforcement Learning — GenJAX stencil

**Human and Machine Learning · Chiba Institute of Technology, SDS**

You will implement **Q-learning** on the *GardenPath* gridworld, then use it to see how the *reward you write* can diverge from the *goal you intend* — and how to fix it.

- **Problem 1** — implement the Q-learning update; confirm it solves the task.
- **Problem 2** — diagnose a reward-hacking failure (a feedback scheme that makes the agent loop forever).
- **Problem 3** — design *and verify* a potential-based-shaping fix.
- **Bonus** — compare your model-free learner to model-based value iteration.

**Textbook:** Chapter 21 (MDPs) and Chapter 22 (Q-learning) of the [Probability Tutorial](https://josephausterweil.github.io/probintro/), including the live Q-learning widget — this assignment uses the same GardenPath world and the same pictures.

**Submit** your completed notebook (run end-to-end, with figures, code, and your written answers) **or** a single PDF with the same content.

*This is the GenJAX stencil: the environment is written as a `@gen` generative model and Q-learning learns by sampling it. A plain-Python stencil (`rl_python.ipynb`) and an R stencil (`rl_nosoln.Rmd`) cover the same problems.*

## Setup

In [ ]:
# Run once on first Colab launch:
# !pip install genjax

In [ ]:
# --- Setup: get the shared GardenPath module (env + visualization) ---
# rl_gridworld.py defines the world, the reward schemes, the static figure
# renderer plot_gridworld(), and the interactive explorer interactive_gridworld().
import os, urllib.request
if not os.path.exists("rl_gridworld.py"):
    try:
        urllib.request.urlretrieve("https://raw.githubusercontent.com/henkaku-center/hml/main/course/assignments/rl/rl_gridworld.py", "rl_gridworld.py")
        print("downloaded rl_gridworld.py")
    except Exception as e:
        print("Could not auto-download rl_gridworld.py:", e)
        print("Upload rl_gridworld.py next to this notebook (Colab: Files tab > upload).")

import numpy as np
import matplotlib.pyplot as plt
import rl_gridworld as G
%matplotlib inline

import jax.numpy as jnp
import jax.random as random
from genjax import gen, categorical

## The environment as a generative function

In GenJAX we write the environment's **transition** $s' \sim P(\cdot\mid s, a)$ as a `@gen` model — a probabilistic program (Chapters 21–22). Here the GardenPath move is deterministic, so the model just puts all its mass on the cell you step into; but the *same* code would handle a stochastic 'slippery' world by spreading that mass out.

Q-learning then learns **by sampling this model**: each step's next state comes from `transition.simulate(...)`, not from a lookup table.

In [ ]:
# Build the (action, state, next-state) transition tensor from the world,
# then write the transition as a GenJAX generative model.
T, idx, inv = G.transition_tensor()      # T[a, s, s'] one-hot; idx/inv map states<->indices
Tj = jnp.array(T)

@gen
def transition(s_idx, a_idx):
    # categorical takes log-probabilities; row Tj[a, s] is the next-state distribution
    return categorical(jnp.log(Tj[a_idx, s_idx] + 1e-12)) @ "s_next"

_key = random.PRNGKey(0)
def genjax_step(s, a):
    """Next state for (state-tuple s, action-char a), drawn from the @gen model."""
    global _key
    _key, k = random.split(_key)
    s2_idx = int(transition.simulate(k, (idx[s], G.ACTIONS.index(a))).get_retval())
    return inv[s2_idx]

# sanity: sampling the model reproduces the deterministic move
print([genjax_step((1,1), a) for a in G.valid_actions((1,1))])
print('matches G.step:', all(genjax_step(s,a) == G.step(s,a)
                             for s in G.states() for a in G.valid_actions(s)))

> **Note.** Drawing each step from the model costs ~4 ms (≈10 s per 2500 steps) — much slower than a numpy lookup. That is fine for a one-time training run here; Chapter 22 shows the *vectorized* `lax.scan` version used for speed at scale.

## Problem 1 — Implement Q-learning, then watch it learn

Q-learning keeps a table $Q(s,a)$ of the expected future reward of taking action $a$ in state $s$. After taking $a$ in $s$, getting reward $r$, and landing in $s'$, it nudges $Q(s,a)$ toward a **target** built from $r$ and the best next value:

$$Q(s,a)\;\leftarrow\;Q(s,a) + \alpha\,\underbrace{\big(r + \gamma\,\max_{a'}Q(s',a') - Q(s,a)\big)}_{\text{prediction error}}$$

**Your task:** fill in the one-line *prediction error* below. `G.q_max(Q, s_next)` returns $\max_{a'}Q(s',a')$ (and 0 at the goal).

In [ ]:
def td_update(Q, s, a, r, s_next, alpha, gamma):
    """Q-learning temporal-difference update, in place."""
    pred_error = None   # <-- FILL ME IN  (one line; see the equation above)
    Q[s][a] += alpha * pred_error
    return pred_error

Train under **reward-maximizing (RM)** feedback — the outcome-based reward (avoid the garden, +20 at the goal) — and look at the learned policy. Green/red wedges are the action-values; white arrows are the greedy policy; the thick line is the route the greedy policy takes from the start (**green = reaches the goal**, red = loops).

In [ ]:
Q_rm = G.fresh_Q()
G.train(Q_rm, G.reward_rm(), n_steps=2500, alpha=0.9, gamma=0.95, eps=0.1,
        td_update=td_update, seed=0, step_fn=genjax_step)
G.plot_gridworld(Q_rm, G.reward_rm()); plt.show()
print(G.verdict(Q_rm, G.reward_rm())[0])

### Explore it interactively
Step through the algorithm one stage at a time, or train in bulk, and watch **your** `td_update` drive the Q-table. (Live in Jupyter/Colab; it will not appear in a static PDF export — use the figure above for your write-up.)

In [ ]:
G.interactive_gridworld(td_update=td_update, scheme='rm')

**Deliverable.** Include the RM figure and one or two sentences: does the learned policy reach the goal, and does it make sense?

## Problem 2 — Diagnose a reward-hacking failure

People rarely give pure outcome rewards; they give **per-step encouragement**. The *action-feedback (AF)* scheme praises forward progress (+10) and — being encouraging — gives only faint praise (+4), not punishment, for backtracking. Train under AF and see what the agent actually learns.

In [ ]:
Q_af = G.fresh_Q()
G.train(Q_af, G.reward_af(), n_steps=2500, td_update=td_update, seed=0,
        step_fn=genjax_step)
G.plot_gridworld(Q_af, G.reward_af()); plt.show()
roll = G.greedy_rollout(Q_af, G.reward_af())
print('reaches goal?', roll['reached'])
print('net reward per lap of the loop:', roll['net_lap'])

**Answer these (a few sentences):**

1. Does the learned greedy policy reach the goal? What does it do instead?
2. Report the **net reward per lap** of the loop. Why does a reward-maximizing agent prefer farming that loop to finishing at the goal?
3. What does this show about the relationship between the *reward you write* and the *goal you intend*?

## Problem 3 — Design and verify a fix

AF *tried* to give helpful dense feedback but created a cycle. The principled way to add dense feedback is **potential-based shaping** (Ng, Harada & Russell 1999): pick a **potential** $\Phi(s)$ (a rough 'how good is this state') and add a potential *difference* to the true reward:

$$r'(s,a) \;=\; r_{\text{RM}}(s,a) \;+\; \gamma\,\Phi(s') - \Phi(s).$$

Because it is a *difference* of potentials, a step toward the goal earns a bonus and a step away pays an equal penalty — so there is no farmable cycle.

**Your task:** design $\Phi(s)$ (a good default: the negative distance to the goal, so states nearer home have higher potential — but try your own and justify it).

In [ ]:
def phi(s):
    """Your potential function. Higher = better (closer to the goal)."""
    return None   # <-- FILL ME IN, e.g.  -G.manhattan_to_goal(s)

shaped = G.reward_shaped(G.reward_rm(), phi, gamma=0.95)
Q_sh = G.fresh_Q()
G.train(Q_sh, shaped, n_steps=4000, td_update=td_update, seed=0)
G.plot_gridworld(Q_sh, shaped); plt.show()
print(G.verdict(Q_sh, shaped)[0])

### Verify the fix is *safe* (invariance)

Shaping should fix learning **without** changing which policy is optimal. Check it on a reward that already works (RM): compute the exact optimal policy with and without your shaping and confirm they match.

In [ ]:
star_rm     = G.q_value_iteration(G.reward_rm(), 0.95)
star_shaped = G.q_value_iteration(G.reward_shaped(G.reward_rm(), phi, 0.95), 0.95)
pol_rm, pol_sh = G.greedy_policy(star_rm), G.greedy_policy(star_shaped)
print('plain-RM  optimal policy:', pol_rm)
print('shaped-RM optimal policy:', pol_sh)
print('same optimum?', pol_rm == pol_sh)

**Answer these (a few sentences):**

1. Does your shaped reward reach the goal?
2. Did shaping change the optimal policy? Explain **why** $\gamma\Phi(s')-\Phi(s)$ cannot change the optimum (hint: sum it along a trajectory — what telescopes?).
3. Contrast this with AF: both add dense feedback, but only one changes the optimum. Why?

## Bonus (+5) — model-based vs model-free

Q-learning is **model-free**: it never knows the transition rules, it learns only from samples. If we *do* know the model (Chapter 21), we can compute the optimal policy exactly with **value iteration**. Confirm your model-free learner found the same route the model-based planner did.

In [ ]:
star = G.q_value_iteration(G.reward_rm(), 0.95)   # exact optimum from the known model
print('value-iteration route:', G.greedy_rollout(star, G.reward_rm())['path'])
print('Q-learning route     :', G.greedy_rollout(Q_rm, G.reward_rm())['path'])